In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import joblib
#import funciones_tesis as ft

## Acá está la clase con la red entrenada

Correr esto para poder definir el modelo.\
La red está entrenada con esta grilla:\
https://github.com/PedroRozin/Tesis2025/tree/main/outputs_pedro/grillas/sin_As

In [7]:
class RegressionNN(nn.Module):
    """ Neural network for regression with 4 input features and 2 output targets.
        Está pensada originalmente para que los features sean: a, Omega_m, kh, h y los targets delta_m y delta_prime_m.
        Architecture:
        - Input layer: 4 neurons (features)
        - Hidden layers: 4 layers with 128 neurons each, ReLU activation
        - Hidden layer: 1 layer with 64 neurons, ReLU activation
        - Output layer: 2 neurons (targets)
    """
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)   # salida de 2 targets
        )
        
    def forward(self, x):
        return self.network(x)

## Condiciones iniciales a partir de la red
Hay que descargarse la carpeta entera y cambiar los paths que sean necesarios (en teoría, solo se debería cambiar 'path_pedro' al path donde tengan descargada la carpeta 'para_javi').

### Cargar modelo y scalers

In [15]:
"""cargar modelo entrenado y los scalers"""


#cambiar paths a donde te descargues la carpeta
path_pedro =  f'/home/pedrorozin/scripts/outputs_pedro/neural_networks/'  #este es mi path base hasta la carpeta donde tengo las redes
_folder_path = 'para_javi'  #carpeta donde está la red para javi
network_name = '_para_javi' #nombre original que le puse a la red

folder_path = f'{path_pedro}/{_folder_path}'
#paths to load model, scalers and (optionally) training history and final metrics


path_model = f'{folder_path}/regression_model{network_name}.pth'
path_training_history = f'{folder_path}/training_history{network_name}.csv' #opcional
path_final_metrics = f'{folder_path}/final_metrics{network_name}.csv' #opcional
path_scaler_X = f'{folder_path}/scaler_X{network_name}.pkl'
path_scaler_y = f'{folder_path}/scaler_y{network_name}.pkl'


# cargar modelo entrenado y los scalers
model = RegressionNN()
model.load_state_dict(torch.load(path_model))  
model.eval()
scaler_X = joblib.load(path_scaler_X)
scaler_y = joblib.load(path_scaler_y)

features = ["a", "k h", "h", "Omega_m"] #la red está entrenada con estos features. notar que k h es kh (en h/Mpc)

### Predicción de UN ÚNICO PAR de condiciones iniciales dado un ÚNICO conjunto de parámetros.

In [16]:
"""obtengo una predicción para un único conjunto de parámetros {a, kh, h, Omega_m}"""

parametros_random = [0.03, 0.01, 0.68, .3] #a_ini, kh (h/Mpc), h, Omega_m
#manera muy ineficiente de hacer la predicción:
a_ini, kh, h, Omega_m = parametros_random
X_single = np.array([[a_ini, kh, h, Omega_m]])

# escalar con los scalers que importamos
X_single_scaled = scaler_X.transform(X_single)
X_single_tensor = torch.tensor(X_single_scaled, dtype=torch.float32)

# evaluar en la red
with torch.no_grad():
    y_pred_scaled = model(X_single_tensor).numpy()

# desescalar
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# ver los de delta_m y delta_prime_m
delta_m_pred, delta_prime_m_pred = y_pred[0, 0], y_pred[0, 1]
print(f"delta_m_pred: {delta_m_pred}")
print(f"delta_prime_m_pred: {delta_prime_m_pred}")

delta_m_pred: -87.05249786376953
delta_prime_m_pred: -2829.6123046875


### Predicción de varios puntos
A la red le podemos pasar varios conjuntos de parámetros {$a$, $kh$, $h$, $\Omega_m$} para que prediga $\delta_m$ y $\delta'_m$. En particular, le podemos pasar todos los puntos de la grilla con la que entrenamos la red.\
**PARA CORRER ESTO HAY QUE TENER DESCARGADA LA GRILLA**

In [27]:
path_data = '/home/pedrorozin/scripts/outputs_pedro/grillas/sin_As/grilla_results_no_As.csv'

df = pd.read_csv(path_data)

features = ["a", "k h", "h", "Omega_m"]
X = df[features].values
targets = ["delta_m", "delta_prime_m"]
y = df[targets].values

# Scaleo las features X usando el mismo escalador del entrenamiento
X_scaled = scaler_X.transform(X)  # Solo transform, NO fit_transform
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32) #esto no se usa xq vamos a comparar directamente con la grilla

# predicción con el modelo (esto da resultados escalados)
with torch.no_grad():
    model.eval()
    y_pred_scaled = model(X_tensor)
    y_pred_scaled = y_pred_scaled.numpy()

# desescalar las predicciones para obtener valores físicos posta
y_pred_physical = scaler_y.inverse_transform(y_pred_scaled)


# create DataFrame con los resultados físicos (sin escalar)
results = pd.DataFrame(y_pred_physical, columns=["delta_m", "delta_prime_m"])

print("=" * 50)
print("Primeras 5 filas - Red Neuronal (desescalada):")
display(results.head(10))
print("\nPrimeras 5 filas - Grilla (valores posta):")  
display(df[targets].head(10))

Primeras 5 filas - Red Neuronal (desescalada):


,delta_m,delta_prime_m
0,-462.811462,-15209.491211
1,-388.788116,-12717.278320
2,-19.206490,-595.202087
3,-61.150398,-1986.789917
4,-334.377258,-10917.287109
5,-252.915283,-8255.368164
6,-44.135933,-1417.203979
7,-284.783997,-9337.916016
8,-797.873047,-26205.691406
9,-84.281563,-2737.116211



Primeras 5 filas - Grilla (valores posta):


,delta_m,delta_prime_m
0,-465.832581,-15296.221277
1,-393.604483,-12924.689077
2,-30.057584,-962.879250
3,-75.661687,-2467.927068
4,-339.416536,-11134.362974
5,-261.571127,-8577.749990
6,-57.559042,-1868.558593
7,-297.244270,-9736.667394
8,-783.643831,-25649.725378
9,-94.835848,-3087.544130


Acá vemos que, aunque las métricas de entrenamiento digan lo contrario, la red es bastante chota. La red hay que mejorarla y seguro cambie, pero se puede ir jugando mientras tanto. 

In [31]:
diferencia = lambda x, y: np.abs(x - y)/np.abs(y) * 100
mayor_1 = []
for target in targets:
	diffs = diferencia(results[target], df[target])
	print(f"\nDiferencias porcentuales para {target}:")
	print(f"  Media: {diffs.mean():.4f}%")
	print(f"  Mediana: {np.median(diffs):.4f}%")
	print(f"  Mínimo: {diffs.min():.4f}%")
	print(f"  Máximo: {diffs.max():.4f}%")
	# if (diffs > 1).any():
	# 	mayor_1.append(target)
	print(f"  Cantidad de puntos con más de 1% de diferencia: {(diffs > 1).sum()} de {len(diffs)}")
	print(f'  Porcentaje de puntos con más de 1% de diferencia: {(diffs > 1).sum()/len(diffs)*100:.2f}%')

print('')
print(" >:(")


Diferencias porcentuales para delta_m:
  Media: 1.3735%
  Mediana: 0.3462%
  Mínimo: 0.0000%
  Máximo: 38.7768%
  Cantidad de puntos con más de 1% de diferencia: 6431 de 28960
  Porcentaje de puntos con más de 1% de diferencia: 22.21%

Diferencias porcentuales para delta_prime_m:
  Media: 1.4902%
  Mediana: 0.3432%
  Mínimo: 0.0000%
  Máximo: 41.4864%
  Cantidad de puntos con más de 1% de diferencia: 6453 de 28960
  Porcentaje de puntos con más de 1% de diferencia: 22.28%

 >:(
